# 03 - First prior: L2 / Tikhonov

Lecture section: 3.6-3.9  |  Spine term this tutorial changes: the prior $R(x)$

$$\hat{x} = \arg\min_x\ \underbrace{D(Ax, y)}_{\text{data fidelity}} + \underbrace{R(x)}_{\text{prior}}$$

So far the prior has been empty and our reconstructions were unusable: sparse-view CT
is ill-posed, so the tiny singular values of $A$ amplify noise without bound. The cure
is to *add a prior* $R(x)$ that penalizes wild solutions. The simplest possible choice
is the **quadratic (Tikhonov) prior** $R(x) = \tfrac{\lambda}{2}\lVert x \rVert^2$.

Here we keep $D$ = least-squares data fidelity (`L2`) and the physics $A$ = sparse-view CT
fixed, set $R(x)=\tfrac{\lambda}{2}\lVert x\rVert^2$, and solve with **proximal gradient
descent (PGD)**. The single knob $\lambda$ controls a bias-variance trade-off: too small
leaves noise, too large blurs everything away.

In [1]:
import tutorial_common as tc
import deepinv as dinv

tc.set_seed()

deepinv 0.4.1 | torch 2.9.1 | device cpu


## The shared problem (identical in notebooks 3, 4, 5)

Sparse-view CT with only 40 projection angles and Gaussian measurement noise
($\sigma=0.02$). This is the *same* operator $A$ and *same* measurement $y$ we will reuse
when we swap in smarter priors (TV, then a learned denoiser) -- only $R$ changes.

In [2]:
phys = tc.ct_physics(angles=40, sigma=0.02)   # fixed physics A (+ noise) = data fidelity D
x = tc.load_hero(128)                          # ground-truth object x  (1,1,128,128)
y = phys(x)                                     # noisy sinogram y = A x + noise
stepsize = tc.stepsize_for(phys, x)            # 1 / ||A^T A||  ~ 1  (normalize=True)

# A naive linear baseline: filtered back-projection (no prior at all).
x_fbp = phys.fbp(y)
print(f"sinogram y shape : {tuple(y.shape)}")
print(f"stepsize         : {stepsize:.3f}")
print(f"FBP baseline     : PSNR {tc.psnr(x_fbp, x):.2f} dB  (no prior -> noisy)")

sinogram y shape : (1, 1, 182, 40)
stepsize         : 1.000
FBP baseline     : PSNR 15.23 dB  (no prior -> noisy)


## Tikhonov via proximal gradient descent

`optim_builder` assembles the iteration. With `prior=Tikhonov()` and
`params_algo={"lambda": lam, "stepsize": stepsize}` it minimizes
$\tfrac12\lVert Ax-y\rVert^2 + \tfrac{\lambda}{2}\lVert x\rVert^2$. One call = one reconstruction.

In [3]:
def tikhonov(lam, max_iter=100):
    """Build + run the PGD solver for a given regularization strength lambda."""
    model = dinv.optim.optim_builder(
        iteration="PGD",
        data_fidelity=dinv.optim.L2(),          # D(Ax, y) = 0.5 ||Ax - y||^2
        prior=dinv.optim.Tikhonov(),            # R(x)     = 0.5 ||x||^2  (quadratic)
        params_algo={"lambda": float(lam), "stepsize": stepsize},
        max_iter=max_iter,
        verbose=False,
    )
    return model(y, phys)

## Bias-variance sweep over $\lambda$

We scan $\lambda$ across five orders of magnitude. Watch the PSNR rise as the prior starts
to suppress noise, peak, then fall as the prior over-smooths and erases the object.

In [4]:
lambdas = [1e-4, 1e-3, 3e-3, 1e-2, 1e-1, 1.0]   # too-small ... peak ... too-large
recons = {lam: tikhonov(lam) for lam in lambdas}
psnrs = {lam: tc.psnr(recons[lam], x) for lam in lambdas}

best_lam = max(psnrs, key=psnrs.get)
for lam in lambdas:
    flag = "  <- best" if lam == best_lam else ""
    print(f"lambda = {lam:<7g}  PSNR = {psnrs[lam]:5.2f} dB{flag}")
print(f"\nPeak at lambda = {best_lam:g}  ->  PSNR {psnrs[best_lam]:.2f} dB")

lambda = 0.0001   PSNR = 19.31 dB
lambda = 0.001    PSNR = 19.36 dB
lambda = 0.003    PSNR = 19.43 dB  <- best
lambda = 0.01     PSNR = 19.28 dB
lambda = 0.1      PSNR = 16.18 dB
lambda = 1        PSNR = 13.69 dB

Peak at lambda = 0.003  ->  PSNR 19.43 dB


## Panel: too small | best | too large

Under-regularized (left) is still grainy from amplified noise; over-regularized (right)
is a featureless blur. The sweet spot in the middle is the best the *quadratic* prior can do.

In [5]:
small_lam, large_lam = lambdas[0], lambdas[-1]
tc.save_images(
    [x, recons[small_lam], recons[best_lam], recons[large_lam]],
    titles=[
        "x (ground truth)",
        tc.title_psnr(f"too small $\\lambda$={small_lam:g}", recons[small_lam], x),
        tc.title_psnr(f"best $\\lambda$={best_lam:g}", recons[best_lam], x),
        tc.title_psnr(f"too large $\\lambda$={large_lam:g}", recons[large_lam], x),
    ],
    fname="03_lambda_sweep.png",
    suptitle="Tikhonov $R(x)=\\frac{\\lambda}{2}||x||^2$: one knob trades noise against blur",
    figsize=(14, 4),   # wide enough that the per-panel titles don't collide
)

saved /Users/jonathan/Code/deepinv/lecture-tutorials/figures/03_lambda_sweep.png


In [6]:
# PSNR-vs-lambda curve: the classic bias-variance trade-off, with a single peak.
tc.save_curves(
    {"Tikhonov": ([float(l) for l in lambdas], [psnrs[l] for l in lambdas])},
    fname="03_bias_variance.png",
    xlabel="$\\lambda$  (regularization strength)",
    ylabel="PSNR (dB)",
    title="Bias-variance trade-off",
    logy=False,
    logx=True,
    markers=True,
)

saved /Users/jonathan/Code/deepinv/lecture-tutorials/figures/03_bias_variance.png


## Convergence at the best $\lambda$

PGD is a descent method: requesting `compute_metrics=True` returns the cost and the
**iterate change** $\lVert x_{k+1}-x_k\rVert$ (what deepinv calls the "residual") at every
iteration, so we can confirm the solver has actually converged (it $\to 0$).

In [7]:
model = dinv.optim.optim_builder(
    iteration="PGD", data_fidelity=dinv.optim.L2(), prior=dinv.optim.Tikhonov(),
    params_algo={"lambda": float(best_lam), "stepsize": stepsize},
    max_iter=100, verbose=False,
)
_, metrics = model(y, phys, x_gt=x, compute_metrics=True)
residual = metrics["residual"][0]   # list-of-lists -> first (only) batch element
tc.save_curves(
    {"residual": residual},
    fname="03_convergence.png",
    xlabel="iteration", ylabel="iterate change  $||x_{k+1}-x_k||$",
    title=f"PGD convergence ($\\lambda$={best_lam:g})", logy=True,
)

saved /Users/jonathan/Code/deepinv/lecture-tutorials/figures/03_convergence.png


## Why Tikhonov can never give sharp edges

Write $A = U\Sigma V^\top$ (the SVD from notebook 2). The Tikhonov solution has a
**closed form** -- it is *Wiener filtering* / *damped SVD inversion*:

$$\hat{x} = \sum_i \underbrace{\frac{\sigma_i}{\sigma_i^2 + \lambda}}_{\text{filtered } 1/\sigma_i}\,(u_i^\top y)\, v_i
  \quad\Longleftrightarrow\quad
  \text{each component is shrunk by } \frac{\sigma_i^2}{\sigma_i^2 + \lambda}.$$

- For **large** singular values ($\sigma_i^2 \gg \lambda$) the factor is $\approx 1$:
  well-measured, low-frequency content passes through untouched.
- For **small** singular values ($\sigma_i^2 \ll \lambda$) the factor is $\approx
  \sigma_i^2/\lambda \to 0$: the noise-amplifying directions are damped -- this is exactly
  the stabilization that fixes the ill-posedness from notebook 2.

But sharp **edges** live precisely in those small-singular-value (high-frequency)
directions. Tikhonov *throws them away*. A quadratic penalty inevitably smooths -- it
denoises but cannot draw a crisp boundary. That is the wall a smarter prior must break.

## Takeaway

A quadratic prior $R(x)=\tfrac{\lambda}{2}\lVert x\rVert^2$ stabilizes the ill-posed
small-singular-value directions and gives us *one knob* $\lambda$ trading noise against
blur -- but because it shrinks **every** component, it smears edges and can never be sharp.
Next we replace it with a prior that *prefers* edges: TV / sparsity.